In [2]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

c:\Users\NTR\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain.agents import create_react_agent, AgentExecutor

In [4]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()   #load env file 

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [5]:
search_tool = TavilySearchResults(max_results=2)

In [6]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [46]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'August 4, 2026\n\n#### Industries\n\n### HoneyBook bets on agentic AI to streamline small business operations with its new Claude connector\n\nWorld of Work\n\nAugust 6, 2026\n\n### Zuckerberg details Meta’s personal AI superintelligence strategy\n\nAI and Us\n\nJuly 30, 2026\n\n### Google’s Gemini 3.6 Flash targets enterprise agent token costs\n\nAI in Action\n\nJuly 21, 2026\n\n#### Deep Learning\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud\n\nAI in Action\n\nJune 8, 2026\n\n### China’s AI just mapped its entire renewable energy grid. Here’s why the rest of the world should pay attention\n\nEnvironment & Sustainability\n\nMay 22, 2026\n\n### IBM Research unveils breakthrough analog AI chip for efficient deep learning\n\nAI Market Trends\n\nAugust 11, 2023\n\n## Subscribe [...] Artificial Intelligence\n\nJanuary 31, 2024\n\n#### Machine Learning\n\n### Samsung health AI models analyse weara

In [7]:
# ==========================================
# LLM
# ==========================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
    api_key=GOOGLE_API_KEY
)

In [49]:
response = llm.invoke("when is navratri in 2026 give me date?")
response

AIMessage(content='In 2026, there are two main Navratri festivals (Spring and Autumn):\n\n### 1. Sharad Navratri 2026 (Main / Autumn Navratri)\nThis is the main, most widely celebrated Navratri (known for Garba, Dandiya, and Durga Puja).\n\n* **Start Date:** Sunday, **October 11, 2026** *(Ghatasthapana)*\n* **End Date:** Monday, **October 19, 2026** *(Maha Navami)*\n* **Dussehra / Vijayadashami:** Tuesday, **October 20, 2026**\n\n---\n\n### 2. Chaitra Navratri 2026 (Spring Navratri)\nThis Navratri marks the Hindu New Year and concludes with Ram Navami.\n\n* **Start Date:** Thursday, **March 19, 2026** *(Ghatasthapana)*\n* **End Date / Ram Navami:** Friday, **March 27, 2026**', response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-0d3e71ca-ecf8-494c-bfb9-aee75dc5f656-0')

In [8]:
# ==========================================
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")


c:\Users\NTR\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [9]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [10]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool,get_weather_data]

In [11]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [12]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True  # show like reasoning steps, actions, and observations in the output
    
)

In [13]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "which country more powerful USA or China"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
Action: tavily_search_results_json
Action Input: most powerful country in the world 2024 USA China ranking ranking consensus spot index GFP Lowy US News ranking[{'url': 'https://nationalinterest.org/blog/buzz/ranked-us-military-officially-most-powerful-2024-208664', 'content': '## Global Firepower Releases 2024 Military Strength Rankings: U.S. Tops the List\n\nThe United States topped Global Firepower’s 2024 Military Strength Rankings, which evaluated 145 countries and assessed them based on their respective military capabilities. Since 2006, GFP has released its findings.\n\nThe top five countries included the United States, Russia, China, India and South Korea. The U.S. scored 0.0699 (the lower the better), with Russia following a close lead with 0.0702 and China at 0.0706. Bhutan was at the bottom of the list with a power index of 6.3704.\n\nMore than 60 individual criteria were factored into the rankings, including the number of military troo

In [55]:
response["output"]

'The capital of India is New Delhi. The current weather in New Delhi is Sunny with a temperature of 38°C and a humidity of 34%.'